# ClaimWise Insurance Prediction — 11. PCA Feature Analysis

## 1. Objective

Principal Component Analysis on the **actual ClaimWise features**, to answer the academic questions:

1. how much of the feature variance do the principal components capture,
2. how many components are actually useful,
3. what dimensionality reduction is achievable,
4. what the components are made of (loadings),
5. whether a PCA-reduced feature space models `loss` as well as the original 130 features.

| Item | Location |
|---|---|
| Data | `Dataset/06_split_X_train.csv`, `06_split_X_test.csv` (the existing pipeline output) |
| Target (unchanged) | `loss` — continuous |
| Shared helpers | `Src/academic_common.py` |

> **What PCA does and does not do.** PCA produces *principal components* — new orthogonal linear
> combinations of all original features. It does **not** pick individual original columns the way a
> filter/feature-importance method does. Statements below therefore describe components (and which
> original features a component is *composed of*), never "PCA selected feature X".

The production regression system (notebooks 04-10) is not modified by this analysis.

## 2. Import Libraries

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Walk upwards from the notebook folder until the ClaimWise project is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Src").is_dir():
            return candidate
    raise FileNotFoundError(f"ClaimWise project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "Dataset"
SRC_DIR = PROJECT_ROOT / "Src"
MODELS_DIR = PROJECT_ROOT / "Models"
OUTPUTS_DIR = PROJECT_ROOT / "Outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from academic_common import PCA_DIR, ensure_dirs, save_figure, style_axes
from model_training_utils import load_prepared_data

ensure_dirs()
print("PCA output folder:", PCA_DIR)

PCA output folder: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/PCA


In [3]:
# Src/model_training_utils.py switches matplotlib to the non-interactive Agg backend so the
# command-line scripts can run without a display. Switch back to the inline backend here so
# that every figure below is rendered inside this notebook.
%matplotlib inline

## 3. Load the ClaimWise Data

The same prepared split the production models use: 40,000 training rows × 130 features and
10,000 test rows × 130 features, all numeric, no missing values.

In [4]:
X_train, X_test, y_train, y_test = load_prepared_data()

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("features:", X_train.shape[1])
print("all numeric:", all(pd.api.types.is_numeric_dtype(d) for d in X_train.dtypes))
print("missing values:", int(X_train.isna().sum().sum() + X_test.isna().sum().sum()))
print("first features:", list(X_train.columns[:4]), "| last:", list(X_train.columns[-4:]))

X_train: (40000, 130) | X_test: (10000, 130)
features: 130
all numeric: True
missing values: 0
first features: ['cat1', 'cat2', 'cat3', 'cat4'] | last: ['cont11', 'cont12', 'cont13', 'cont14']


## 4. Scaling Before PCA

PCA maximises variance, so features with larger numeric ranges would dominate. The pipeline already
standardised `cont1`–`cont14`, but the label-encoded `cat*` columns are integer codes with different
spreads — and the encoded integers carry no meaningful metric scale either.

A `StandardScaler` is therefore fitted **on the training rows only** and applied to both splits.
Fitting on training data only keeps the analysis leakage-free: the test rows never influence the
transformation.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("max |train column mean| after scaling:", float(np.abs(X_train_scaled.mean(axis=0)).max()))
print("max |test  column mean| after scaling:", float(np.abs(X_test_scaled.mean(axis=0)).max()))
print("(train is exactly centred; test is only close — that is the expected leakage-free result)")

max |train column mean| after scaling: 1.105782132526656e-15
max |test  column mean| after scaling: 0.03846021144160469
(train is exactly centred; test is only close — that is the expected leakage-free result)


## 5. PCA — Explained Variance

`PCA()` with no component limit keeps all 130 components so the full spectrum is available; the
useful number of components is then read off the cumulative curve.

In [6]:
from sklearn.decomposition import PCA

pca = PCA(random_state=42)
pca.fit(X_train_scaled)

evr = pca.explained_variance_ratio_
cumulative = np.cumsum(evr)

print("components fitted :", len(evr))
print("sum of explained variance ratio: {:.6f}".format(cumulative[-1]))
for target in (0.90, 0.95, 0.99):
    n_needed = int(np.searchsorted(cumulative, target) + 1)
    print(f"components needed for {target:.0%} variance: {n_needed}")

components fitted : 130
sum of explained variance ratio: 1.000000
components needed for 90% variance: 77
components needed for 95% variance: 89
components needed for 99% variance: 104


In [7]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(range(1, len(evr) + 1), evr, color="#D4512A", label="explained variance ratio")
ax.plot(range(1, len(evr) + 1), cumulative, color="#18181B", marker="o", markersize=3,
        linewidth=1.4, label="cumulative")
ax.set_xlim(0, min(len(evr), 60) + 1)
style_axes(ax, "PCA explained variance — first 60 components", "principal component",
           "explained variance ratio")
ax.legend()
path = save_figure(fig, PCA_DIR / "pca_explained_variance_ratio.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/PCA/pca_explained_variance_ratio.png


In [8]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(cumulative) + 1), cumulative, color="#D4512A", linewidth=2)
for target, color in ((0.90, "#78716C"), (0.95, "#18181B"), (0.99, "#E58A3A")):
    n_needed = int(np.searchsorted(cumulative, target) + 1)
    ax.axhline(target, linestyle="--", linewidth=1, color=color)
    ax.axvline(n_needed, linestyle=":", linewidth=1, color=color)
    ax.annotate(
        f"{target:.0%} → {n_needed} components",
        xy=(n_needed, target), xytext=(n_needed + 6, target - 0.06),
        fontsize=10, color=color,
    )
ax.set_xlim(0, len(cumulative) + 1)
style_axes(ax, "Cumulative explained variance vs number of components",
           "number of principal components", "cumulative explained variance")
path = save_figure(fig, PCA_DIR / "pca_cumulative_explained_variance.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/PCA/pca_cumulative_explained_variance.png


In [9]:
rows = []
for n in [1, 2, 3, 5, 10, 15, 20, 30, 40, 50, 75, 100, 130]:
    if n <= len(cumulative):
        rows.append({
            "components": n,
            "cumulative_explained_variance": float(cumulative[n - 1]),
            "reduction_vs_130_features": f"{100 * (1 - n / 130):.1f}%",
        })
components_table = pd.DataFrame(rows)
display(components_table.round(4))

table_path = PCA_DIR / "pca_explained_variance_table.csv"
components_table.to_csv(table_path, index=False)
print("Saved:", table_path)

,components,cumulative_explained_variance,reduction_vs_130_features
0,1,0.0849,99.2%
1,2,0.1358,98.5%
2,3,0.1744,97.7%
3,5,0.2405,96.2%
4,10,0.3604,92.3%
5,15,0.4375,88.5%
6,20,0.4941,84.6%
7,30,0.5886,76.9%
8,40,0.6675,69.2%
9,50,0.7403,61.5%


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/PCA/pca_explained_variance_table.csv


## 6. Loadings — What the Components Are Made Of

A loading is the weight of an original feature inside a principal component. Reading the largest
loadings of PC1/PC2/PC3 tells us *which original features dominate those components* — it does **not**
mean those features were "selected", because every component uses all 130 features.

In [10]:
loadings = pd.DataFrame(
    pca.components_[:3].T,
    index=X_train.columns,
    columns=["PC1", "PC2", "PC3"],
)
top_pc1 = loadings["PC1"].abs().sort_values(ascending=False).head(10)
top_pc1_signed = loadings.loc[top_pc1.index, "PC1"]

print("Top 10 features composing PC1 (signed loadings):")
display(top_pc1_signed.round(4).to_frame("PC1_loading"))

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(top_pc1_signed.index[::-1], top_pc1_signed.values[::-1],
        color=["#D4512A" if v > 0 else "#78716C" for v in top_pc1_signed.values[::-1]])
style_axes(ax, "PC1 loadings — 10 largest contributions", "loading (signed)", "")
path = save_figure(fig, PCA_DIR / "pca_pc1_top_loadings.png")
print("Saved:", path)

Top 10 features composing PC1 (signed loadings):


,PC1_loading
cont6,0.2858
cat104,0.2659
cont10,0.2569
cat116,0.2548
cont13,0.2546
cont9,0.2469
cont12,0.2375
cont11,0.2334
cat86,0.2325
cont1,0.2308


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/PCA/pca_pc1_top_loadings.png


In [11]:
top_pc2 = loadings["PC2"].abs().sort_values(ascending=False).head(10)
print("Top 10 features composing PC2 (signed loadings):")
display(loadings.loc[top_pc2.index, "PC2"].round(4).to_frame("PC2_loading"))

explained_pc = pd.DataFrame({
    "component": ["PC1", "PC2", "PC3"],
    "explained_variance_ratio": evr[:3],
})
display(explained_pc.round(4))

Top 10 features composing PC2 (signed loadings):


,PC2_loading
cat101,0.3114
cat2,0.2765
cat9,0.2661
cat6,-0.2359
cat10,0.2314
cat12,0.2272
cat50,-0.2241
cat114,-0.2166
cat87,0.1993
cat11,0.1908


,component,explained_variance_ratio
0,PC1,0.0849
1,PC2,0.0509
2,PC3,0.0386


## 7. 2D Projection of the Test Rows

Projecting the held-out rows onto the first two components gives a visual impression of the data's
structure. Colouring by `loss` shows how much of the target's variation the first two components
capture — a deliberately modest amount in most real datasets, which is why PC1+PC2 alone are rarely
enough for modelling.

In [12]:
test_projection = pca.transform(X_test_scaled)[:, :2]
n_show = min(4000, len(test_projection))
sample_idx = np.random.RandomState(42).choice(len(test_projection), size=n_show, replace=False)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    test_projection[sample_idx, 0],
    test_projection[sample_idx, 1],
    c=y_test.to_numpy()[sample_idx],
    cmap="viridis", s=10, alpha=0.6,
)
fig.colorbar(scatter, ax=ax, label="loss")
style_axes(ax, f"PCA projection of {n_show} test rows (PC1 vs PC2, coloured by loss)",
           "PC1", "PC2")
path = save_figure(fig, PCA_DIR / "pca_projection_2d.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/PCA/pca_projection_2d.png


## 8. Modelling Experiment — Original Space vs PCA-Reduced Space

Question: does a PCA-reduced feature space predict `loss` as well as the original 130 features?

* **Original space** — the existing 130 pipeline features (production setting).
* **PCA space** — the number of components that explains 95% of the training variance, with the
  scaler and the PCA both fitted on the **training rows only**.

Two models are used: Linear Regression (the baseline) and the Hist Gradient Boosting Regressor (the
current production best model). Same split, same metrics as the production system
(MAE / MSE / RMSE / R²), so the numbers are directly comparable.

In [13]:
from sklearn.linear_model import LinearRegression

from hist_gradient_boosting_regressor_model import build_model as build_hist_gbm

n_95 = int(np.searchsorted(cumulative, 0.95) + 1)
print(f"Components for 95% variance: {n_95} (reduction: {100 * (1 - n_95 / 130):.1f}% vs 130 features)")

from sklearn.pipeline import Pipeline

pca_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=n_95, random_state=42)),
])

experiment_rows = []
predictions = {}
for space, transformer in (("original 130 features", None), (f"PCA ({n_95} components)", pca_pipe)):
    if transformer is None:
        Xtr, Xte = X_train, X_test
    else:
        Xtr = transformer.fit_transform(X_train)
        Xte = transformer.transform(X_test)

    for model_name, factory in (
        ("Linear Regression", lambda: LinearRegression()),
        ("Hist Gradient Boosting Regressor", lambda: build_hist_gbm()),
    ):
        model = factory()
        model.fit(Xtr, y_train)
        pred = model.predict(Xte)
        predictions[(space, model_name)] = pred
        mae = float(np.mean(np.abs(y_test.to_numpy() - pred)))
        mse = float(np.mean((y_test.to_numpy() - pred) ** 2))
        rmse = float(np.sqrt(mse))
        r2 = float(1 - mse / np.var(y_test.to_numpy()))
        experiment_rows.append({
            "feature_space": space,
            "model": model_name,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2,
        })
        print(f"{space:26s} | {model_name:32s} | MAE {mae:8.4f} | RMSE {rmse:8.4f} | R2 {r2:.6f}")

experiment = pd.DataFrame(experiment_rows)
exp_path = PCA_DIR / "pca_modeling_experiment.csv"
experiment.to_csv(exp_path, index=False)
print("Saved:", exp_path)

Components for 95% variance: 89 (reduction: 31.5% vs 130 features)


original 130 features      | Linear Regression                | MAE 1333.5665 | RMSE 2050.5223 | R2 0.482615


original 130 features      | Hist Gradient Boosting Regressor | MAE 1215.2997 | RMSE 1921.0240 | R2 0.545901


PCA (89 components)        | Linear Regression                | MAE 1338.4415 | RMSE 2065.5173 | R2 0.475020


PCA (89 components)        | Hist Gradient Boosting Regressor | MAE 1270.5404 | RMSE 1995.6963 | R2 0.509912
Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/PCA/pca_modeling_experiment.csv


In [14]:
pivot = experiment.pivot(index="model", columns="feature_space", values="R2")
ax = pivot.plot(kind="bar", figsize=(9, 4.8), color=["#D4512A", "#18181B"], rot=0)
style_axes(ax.get_figure().axes[0], "R² on the test set — original space vs PCA-reduced space",
           "", "R²")
for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", fontsize=9, padding=2)
ax.set_ylim(0, max(0.65, float(pivot.max().max()) + 0.1))
ax.legend(title="feature space")
path = save_figure(ax.get_figure(), PCA_DIR / "pca_modeling_experiment.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/PCA/pca_modeling_experiment.png


## 9. Conclusion

Actual results of this executed run:

| Question | Answer |
|---|---|
| Components fitted | 130 (all — the full spectrum was fitted so the useful count can be read from the curve) |
| Variance explained by all 130 components | 1.000000 (the 130 components span the whole feature space by construction) |
| Components needed for 95% variance | 89 components (90% → 77, 99% → 104) |
| Dimensionality reduction at 95% | 31.5% fewer dimensions (130 → 89) |
| Best original-space model (R²) | original space, Hist Gradient Boosting: R2 0.545901 (MAE 1215.2997) |
| Best PCA-space model (R²) | PCA space, Hist Gradient Boosting: R2 0.509912 (MAE 1270.5404) |
| PCA better than the original space? | No — PCA (89 components) is worse on both models (HistGB R2 0.509912 vs 0.545901, Linear R2 0.475020 vs 0.482615), so the production model keeps the original 130 features and PCA stays an analysis/modeling experiment |

Interpretation: PCA is a **dimensionality-reduction / analysis** technique. It produced components
(and their loadings), not a subset of original features. The production ClaimWise model keeps the
original 130-feature space unless the experiment above had clearly justified a change.

Artifacts: `Outputs/PCA/pca_explained_variance_ratio.png`, `pca_cumulative_explained_variance.png`,
`pca_pc1_top_loadings.png`, `pca_projection_2d.png`, `pca_explained_variance_table.csv`,
`pca_modeling_experiment.csv`, `pca_modeling_experiment.png`.